# Circuit-tracing feasibility check: Qwen3-1.7B

**Question:** can we run `circuit-tracer` (attribution graphs from per-layer transcoders) on Qwen3-1.7B, on our GSM8K cases, with the compute we have? This is a feasibility check, not an experiment. Nothing here is a finding about faithfulness. Cases A and B come from the 1.7B model's saved outputs, so tracing them through the 0.6B model does not reproduce the original model's internals; with `MODEL_SIZE = "0.6B"` this is a pipeline feasibility test only.

**How to run (Colab):**
1. Runtime → Change runtime type → **A100 GPU** (needs Colab Pro) if you can. **L4** is the next best. **T4 (free)** is a long shot, see "Memory" below.
2. Run cell 1 (install). If Colab asks to restart the session, do it, then **continue from cell 2** (do not re-run cell 1).
3. Run all remaining cells top to bottom. Cell 5 asks you to upload `qwen3_1.7b_gsm8k.jsonl` from `outputs/raw/` if it can't find it. It's only read, never modified.
4. Last cell zips `outputs/clean/circuit_tracing/` and downloads it. Unzip it into the repo at the same path.

**Memory (read this before picking a GPU).** The 1.7B transcoder set (`mwhanna/qwen3-1.7b-transcoders-lowl0`) has 163,840 features per layer × 28 layers, stored in bf16 as ~37.6 GB of `layer_*.safetensors` (checked on the Hugging Face repo). The encoders alone are ~18.8 GB in bf16. That does not fit next to the model on a 15 GB T4, so the notebook turns on `lazy_encoder` for GPUs under 32 GB. That is slower and **untested** for this set. Expect a ~38 GB download on first load.

**What was not verified before this was written:** nothing in this notebook has been executed. API calls were taken from the library's source and demo notebooks (`attribute_demo.ipynb`, `attribution_targets_demo.ipynb`), not from a run.

**Prompt format and generation settings** are copied from `milestone1.ipynb` below, for reference. This notebook does **not** generate text. It replays saved outputs from `outputs/raw/` and attributes the next-token prediction at the final-answer position.

### 1. Install (restart the Colab session if it asks, then continue from cell 2)

In [ ]:
# Follows the install steps in the library's demo notebooks (clone + editable install).
# The library pins transformers<=4.57.3 and nnsight 0.8.x, so this may replace Colab's preinstalled versions.
import os
if not os.path.exists("circuit-tracer"):
    !git clone -q https://github.com/decoderesearch/circuit-tracer
%pip install -q -e circuit-tracer

### 2. Config and environment log

In [ ]:
MODEL_SIZE = "0.6B"   # "0.6B" or "1.7B"; the only knob for model choice. 1.7B crashed (out of CPU RAM) on free Colab at model load.
assert MODEL_SIZE in ("0.6B", "1.7B"), MODEL_SIZE

import sys, os, re, json, time, shutil, traceback, platform, importlib.metadata as md_
from pathlib import Path
import torch

IN_COLAB = "google.colab" in sys.modules

# ---- model / transcoders (derived from MODEL_SIZE; both repos verified to exist on the Hugging Face Hub and listed in the library README) ----
MODEL_NAME = f"Qwen/Qwen3-{MODEL_SIZE}"
TRANSCODER_SET = f"mwhanna/qwen3-{MODEL_SIZE.lower()}-transcoders-lowl0"
BACKEND = "transformerlens"   # library default. If loading fails, try "nnsight" (experimental, slower).

# ---- copied from milestone1.ipynb (reference only; this notebook does not generate) ----
GENERATION_CONFIG_THINKING_ON  = {"temperature": 0.6, "top_p": 0.95, "top_k": 20, "max_new_tokens": 2048}
GENERATION_CONFIG_THINKING_OFF = {"temperature": 0.7, "top_p": 0.8,  "top_k": 20, "max_new_tokens": 1024}
GENERATION_CONFIG_TRUNCATED_RERUN = {"temperature": 0.6, "top_p": 0.95, "top_k": 20, "max_new_tokens": 8192}

# ---- attribution settings (names/defaults from attribute_demo.ipynb) ----
MAX_FEATURE_NODES = 4096      # demo uses 8192; lower = faster/smaller, loses more of the graph
BATCH_SIZES_TO_TRY = (128, 64, 32)   # retried in order if a run hits CUDA OOM (demo default is 256)
OFFLOAD = "disk" if IN_COLAB else "cpu"   # demo's choice; can be 'cpu', 'disk' or None
NODE_THRESHOLD, EDGE_THRESHOLD = 0.8, 0.98   # create_graph_files defaults

OUT_DIR = Path("outputs/clean/circuit_tracing")
GRAPH_FILE_DIR = OUT_DIR / "graph_files"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> pick a GPU."
props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / 2**30
BF16_OK = props.major >= 8          # T4 is 7.5 -> no native bf16
DTYPE = torch.bfloat16 if BF16_OK else torch.float16   # milestone1 used float16 on T4
LAZY_ENCODER = VRAM_GB < 32         # encoders alone are ~18.8 GB in bf16 at 1.7B (about half that at 0.6B, from file sizes; threshold not tuned per size)
LAZY_DECODER = True                 # library default

def _ver(pkg):
    try: return md_.version(pkg)
    except Exception: return "unknown"

ENV = {
    "model_size": MODEL_SIZE, "model_name": MODEL_NAME, "transcoders": TRANSCODER_SET,
    "gpu": props.name, "vram_gb": round(VRAM_GB, 1), "compute_capability": f"{props.major}.{props.minor}",
    "dtype": str(DTYPE), "lazy_encoder": LAZY_ENCODER, "lazy_decoder": LAZY_DECODER,
    "python": platform.python_version(), "torch": torch.__version__,
    "transformers": _ver("transformers"), "transformer_lens": _ver("transformer-lens"),
    "nnsight": _ver("nnsight"), "circuit_tracer": _ver("circuit-tracer"),
    "disk_free_gb": round(shutil.disk_usage("/").free / 2**30, 1),
}
try:
    import psutil; ENV["ram_gb"] = round(psutil.virtual_memory().total / 2**30, 1)
except Exception: pass
print(json.dumps(ENV, indent=2))
if ENV["disk_free_gb"] < 45:
    print("WARNING: <45 GB free disk; the transcoder download is ~38 GB.")

In [ ]:
# Pre-flight (run before the load cell): transcoder repo size on the Hub vs. this machine's free disk and RAM.
from huggingface_hub import HfApi
import psutil
info = HfApi().model_info(TRANSCODER_SET, files_metadata=True)
sizes = {s.rfilename: (s.size or 0) for s in info.siblings}
total_gb = sum(sizes.values()) / 1e9
weights_gb = sum(v for k, v in sizes.items() if k.endswith(".safetensors")) / 1e9
print(f"{TRANSCODER_SET}: {len(sizes)} files, total {total_gb:.1f} GB (layer_*.safetensors alone: {weights_gb:.1f} GB)")
vm = psutil.virtual_memory()
print(f"free disk: {shutil.disk_usage('/').free / 1e9:.1f} GB | RAM total {vm.total / 1e9:.1f} GB, available {vm.available / 1e9:.1f} GB")

### 3. Logging helpers

In [ ]:
import gc, csv
RUN_LOG = []   # one dict per run (model load, then each attribution)

def gpu_gb(): return torch.cuda.memory_allocated() / 2**30

def cleanup():
    gc.collect(); torch.cuda.empty_cache()

def save_log():
    (OUT_DIR / "run_log.json").write_text(json.dumps({"env": ENV, "runs": RUN_LOG}, indent=2, default=str))
    keys = sorted({k for r in RUN_LOG for k in r})
    with open(OUT_DIR / "run_log.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); [w.writerow(r) for r in RUN_LOG]

### 4. Load model and transcoders

In [ ]:
import os, sys
print("repo folder exists:", os.path.exists("circuit-tracer"))
print("package folder exists:", os.path.exists("circuit-tracer/circuit_tracer"))
sys.path.insert(0, os.path.abspath("circuit-tracer"))
import circuit_tracer
print("import ok:", circuit_tracer.__file__)

In [ ]:
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
from circuit_tracer.utils.demo_utils import cleanup_cuda

rec = {"run": "load_model", "model_name": MODEL_NAME, "model_size": MODEL_SIZE, "transcoders": TRANSCODER_SET,
       "backend": BACKEND, "dtype": str(DTYPE), "lazy_encoder": LAZY_ENCODER, "lazy_decoder": LAZY_DECODER}
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
try:
    model = ReplacementModel.from_pretrained(
        MODEL_NAME, TRANSCODER_SET, backend=BACKEND, dtype=DTYPE,
        lazy_encoder=LAZY_ENCODER, lazy_decoder=LAZY_DECODER,
    )
    rec["status"] = "ok"
except Exception as e:
    rec["status"] = "error"; rec["error"] = f"{type(e).__name__}: {e}"; rec["traceback"] = traceback.format_exc()
    print(rec["traceback"])
rec["runtime_s"] = round(time.time() - t0, 1)
rec["gpu_mem_after_load_gb"] = round(gpu_gb(), 2)
rec["gpu_peak_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
RUN_LOG.append(rec); save_log()
print({k: v for k, v in rec.items() if k != "traceback"})
assert rec["status"] == "ok", "Model load failed, see the error above (logged in run_log.json)."
tok = model.tokenizer

### 5. Load cases from `outputs/raw/`

In [ ]:
RAW_NAME = "qwen3_1.7b_gsm8k.jsonl"
cands = [Path("outputs/raw")/RAW_NAME, Path("../../outputs/raw")/RAW_NAME, Path(RAW_NAME)]
raw_path = next((p for p in cands if p.exists()), None)
if raw_path is None:
    from google.colab import files
    print(f"Upload {RAW_NAME} (from outputs/raw/) ...")
    up = files.upload(); raw_path = Path(next(iter(up)))
rows = [json.loads(l) for l in open(raw_path)]   # read-only
print(f"{len(rows)} rows from {raw_path}")

def last_row(id_, mode):
    # raw file has re-run rows appended; take the last non-empty one for this (id, mode)
    m = [r for r in rows if r["id"] == str(id_) and r["mode"] == mode and (mode == "no_thinking" or r["thinking"])]
    return m[-1]

def final_answer(text):
    m = re.findall(r"Final answer:\s*\**\$?\s*(\d[\d,\.]*)", text, re.I)
    return m[-1].replace(",", "").rstrip(".") if m else None

# Chosen after reading the raw file (see journal entry); by eye, not by a formal check:
#  A: thinking id 11, gold 694. Reasoning reaches 694 ("Final answer: 694" at the end of the thinking),
#     but the response reads "Final answer: <number>" (the template placeholder), so parsed model_answer is None.
#     This is the closest thing to "answer does not follow reasoning" in the pilot data.
#  B: thinking id 7, gold 160, answer 120. A wrong answer whose reasoning supports 120 (consistent, not a mismatch).
#     It is also the longer trace, so it doubles as a memory stress test.
for name, (i, mode) in {"A (id 11, thinking)": (11, "thinking"), "B (id 7, thinking)": (7, "thinking")}.items():
    r = last_row(i, mode)
    print(name, "| gold:", r["gold_answer"], "| parsed answer:", repr(r["model_answer"]),
          "| answer in reasoning:", final_answer(r["thinking"]), "| answer in response:", final_answer(r["response"]))

### 6. Prompt construction (same user message as `milestone1.ipynb`)

In [ ]:
def user_message(question):
    return f"{question}\nSolve this step by step. End your response with 'Final answer: <number>'."

def chat_prefix(question, enable_thinking):
    return tok.apply_chat_template([{"role": "user", "content": user_message(question)}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking)

ANSWER_STUB = "Final answer: "   # prefix stops here; the target is the next token (first digit of the answer)

def thinking_prefix(row):
    # chat prompt + the full saved thinking + </think> + "Final answer: ". The response body is dropped.
    think = row["thinking"].strip()
    if not think.startswith("<think>"): think = "<think>\n" + think
    return chat_prefix(row["question"], True) + think + "\n</think>\n\n" + ANSWER_STUB

def first_digit_target(answer):
    # Qwen tokenises digits one at a time, so the 'answer token' is the first digit.
    ids = tok.encode(answer[0], add_special_tokens=False)
    assert len(ids) == 1, ids
    return ids[0], answer[0]

### 7. Run function (logs model, tokens, GPU memory, runtime, errors)

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

def run_case(slug, prefix, answer_for_target, note=""):
    target_id, target_str = first_digit_target(answer_for_target)
    rec = {"run": slug, "note": note, "model_name": MODEL_NAME, "model_size": MODEL_SIZE, "transcoders": TRANSCODER_SET,
           "dtype": str(DTYPE), "target_token": target_str, "target_id": target_id, "answer_for_target": answer_for_target,
           "max_feature_nodes": MAX_FEATURE_NODES, "offload": OFFLOAD, "attempts": []}
    ids = model.ensure_tokenized(prefix)
    rec["n_tokens"] = int(ids.shape[-1])
    print(f"[{slug}] {rec['n_tokens']} tokens | tail: {tok.decode(ids.reshape(-1)[-12:])!r}")

    # What does the model itself predict at this position? (no attribution yet)
    with torch.no_grad():
        logits, _ = model.get_activations(ids)
    last = logits[0, -1] if logits.dim() == 3 else logits[-1]
    probs = torch.softmax(last.float(), -1)
    top = torch.topk(probs, 5)
    rec["model_top5"] = [(tok.decode([int(i)]), round(float(p), 4)) for p, i in zip(top.values, top.indices)]
    rec["p_target"] = round(float(probs[target_id]), 4)
    print(f"[{slug}] model top-5: {rec['model_top5']} | p(target {target_str!r}) = {rec['p_target']}")
    del logits, last, probs; cleanup_cuda()

    graph = None
    for bs in BATCH_SIZES_TO_TRY:
        att = {"batch_size": bs}
        torch.cuda.reset_peak_memory_stats(); t0 = time.time()
        try:
            graph = attribute(
                prompt=prefix, model=model,
                attribution_targets=torch.tensor([target_id]),   # final-answer token only (no salient-logit search)
                batch_size=bs, max_feature_nodes=MAX_FEATURE_NODES, offload=OFFLOAD, verbose=True,
            )
            att["status"] = "ok"
        except torch.cuda.OutOfMemoryError as e:
            att["status"] = "oom"; att["error"] = str(e)[:300]
        except Exception as e:
            att["status"] = "error"; att["error"] = f"{type(e).__name__}: {e}"; att["traceback"] = traceback.format_exc()
        att["runtime_s"] = round(time.time() - t0, 1)
        att["gpu_peak_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
        rec["attempts"].append(att); print(f"[{slug}] attempt {att}")
        if att["status"] == "ok": break
        cleanup_cuda()
        if att["status"] == "error": break   # only retry on OOM

    last_att = rec["attempts"][-1]
    rec.update(status=last_att["status"], batch_size=last_att["batch_size"],
               runtime_s=last_att["runtime_s"], gpu_peak_gb=last_att["gpu_peak_gb"])
    if graph is not None:
        pt = OUT_DIR / f"{slug}.pt"
        graph.to_pt(pt)
        rec["graph_pt_mb"] = round(pt.stat().st_size / 2**20, 1)
        rec["n_active_features"] = int(graph.active_features.shape[0])
        rec["adjacency_shape"] = list(graph.adjacency_matrix.shape)
        rec["n_logit_targets"] = len(graph.logit_targets)
        try:
            create_graph_files(graph_or_path=str(pt), slug=slug, output_path=str(GRAPH_FILE_DIR),
                               node_threshold=NODE_THRESHOLD, edge_threshold=EDGE_THRESHOLD)
            rec["graph_files"] = "ok"
        except Exception as e:
            rec["graph_files"] = f"error: {type(e).__name__}: {e}"
        # simple summary figure from documented fields: active features per layer (active_features rows are (layer, pos, feature))
        layers = graph.active_features[:, 0].cpu()
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.bar(*torch.unique(layers, return_counts=True)); ax.set_xlabel("layer"); ax.set_ylabel("# active features")
        ax.set_title(f"{slug}: {rec['n_tokens']} tokens, target {target_str!r}")
        fig.tight_layout(); fig.savefig(OUT_DIR / f"{slug}_active_features_by_layer.png", dpi=150); plt.close(fig)
    del graph; cleanup_cuda()
    RUN_LOG.append(rec); save_log()
    print({k: v for k, v in rec.items() if k not in ("attempts",)})
    return rec

### 8. Sanity test: one short GSM8K problem (thinking off)
GSM8K test problem 0 with its reference solution as the "reasoning" (so the prefix is short). Target: first digit of the gold answer, `18` → `1`.

In [ ]:
from datasets import load_dataset
ds0 = load_dataset("openai/gsm8k", "main", split="test")[0]
solution = re.sub(r"<<.*?>>", "", ds0["answer"].split("####")[0]).strip()
gold = ds0["answer"].split("####")[1].strip()
assert ds0["question"] == last_row(0, "no_thinking")["question"]   # same problem as the pilot's id 0

sanity_prefix = chat_prefix(ds0["question"], False) + "Let's solve this step by step.\n\n" + solution + "\n\n" + ANSWER_STUB
sanity = run_case("sanity_gsm8k0_nothink", sanity_prefix, gold, note="short; reference solution as reasoning")

### 9. Case A: reasoning reaches 694 but the response says `<number>` (thinking id 11)

In [ ]:
rowA = last_row(11, "thinking")
caseA = run_case("caseA_id11_thinking", thinking_prefix(rowA), final_answer(rowA["thinking"]),
                 note="target = answer the reasoning supports (694 -> '6'); compare model_top5 to see what the model actually predicts here")

### 10. Case B: wrong answer (120 vs gold 160), longer trace (thinking id 7)

In [ ]:
rowB = last_row(7, "thinking")
caseB = run_case("caseB_id7_thinking", thinking_prefix(rowB), final_answer(rowB["thinking"]),
                 note="wrong answer, reasoning-consistent; longest trace of the three, memory stress test")

### 11. Summary, save, download

In [ ]:
import pandas as pd
df = pd.DataFrame([{k: r.get(k) for k in ["run", "status", "model_size", "n_tokens", "batch_size", "runtime_s",
                                          "gpu_peak_gb", "gpu_mem_after_load_gb", "n_active_features",
                                          "adjacency_shape", "graph_pt_mb", "p_target", "graph_files"]} for r in RUN_LOG])
display(df)
save_log()
for r in RUN_LOG:
    for a in r.get("attempts", []):
        if a.get("error"): print(r["run"], a["status"], a["error"][:200])

if IN_COLAB:
    shutil.make_archive("circuit_tracing_outputs", "zip", OUT_DIR)
    from google.colab import files
    files.download("circuit_tracing_outputs.zip")   # unzip into outputs/clean/circuit_tracing/ in the repo

### 12. (Optional) Look at a graph in the browser
Uses the library's local server, as in `attribute_demo.ipynb`. Screenshots have to be taken by hand from the iframe (save into `outputs/clean/circuit_tracing/`).

In [ ]:
from circuit_tracer.frontend.local_server import serve
port = 8046
server = serve(data_dir=str(GRAPH_FILE_DIR) + "/", port=port)
if IN_COLAB:
    from google.colab import output as colab_output
    colab_output.serve_kernel_port_as_iframe(port, path="/index.html", height="800px", cache_in_notebook=True)
else:
    print(f"Open http://localhost:{port}/index.html")
# server.stop()   # when done